In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load dataset
df = pd.read_csv(
"seasonal_disease_dataset_500.csv"
)

In [ ]:
# Fill Missing Values
df["Rainfall_mm"].fillna(
df["Rainfall_mm"].median(),
inplace=True
)

In [ ]:
df["Temperature_C"].fillna(
df["Temperature_C"].median(),
inplace=True
)

In [ ]:
df["Humidity"].fillna(
df["Humidity"].median(),
inplace=True
)

In [ ]:
df["OPD_Visits"].fillna(
df["OPD_Visits"].median(),
inplace=True
)

In [ ]:
df["Disease_Name"].fillna(
df["Disease_Name"].mode()[0],
inplace=True
)

In [ ]:
# Remove Outliers
df = df[
df["Rainfall_mm"] < 300
]

df = df[
df["Temperature_C"] < 50
]

In [ ]:
# Outbreak Index
df["Outbreak_Index"] = (
0.3*df["Rainfall_mm"] +
0.2*df["Humidity"] +
0.5*df["OPD_Visits"]
)

In [ ]:
# Date Features

df["Date"] = pd.to_datetime(
df["Date"]
)

df["Month"] = df["Date"].dt.month

In [ ]:
# Rolling Forecast Features
# important for outbreak prediction

df["Rain_7DayAvg"] = (
df["Rainfall_mm"]
.rolling(7)
.mean()
)

df["OPD_7DayAvg"] = (
df["OPD_Visits"]
.rolling(7)
.mean()
)


In [ ]:
# Create Future Risk
# rule-based early warning target

df["Future_Risk"]="Low"

In [ ]:
# High outbreak conditions
df.loc[
(df["Rain_7DayAvg"] > 100) &
(df["Humidity"] > 70) &
(df["OPD_7DayAvg"] > 150),
"Future_Risk"
]="High"

In [ ]:
# Medium outbreak conditions
df.loc[
(df["Temperature_C"] > 34) &
(df["Future_Risk"]!="High"),
"Future_Risk"
]="Medium"

In [ ]:
# Remove NaN from rolling windows
df=df.dropna()

In [ ]:
# Save cleaned data
df.to_csv(
"cleaned_disease_dataset_v2.csv",
index=False
)

In [ ]:
print("Updated dataset ready")

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

In [ ]:
# Load updated data
df = pd.read_csv(
"cleaned_disease_dataset_v2.csv"
)

In [ ]:
# Encode District
df = pd.get_dummies(
df,
columns=["District"]
)

In [ ]:
# Features
feature_cols = [
"Rainfall_mm",
"Temperature_C",
"Humidity",
"OPD_Visits",
"Month",
"Outbreak_Index",
"Rain_7DayAvg",
"OPD_7DayAvg"
]

In [ ]:
# add district columns
for col in df.columns:

    if col.startswith("District_"):

        feature_cols.append(col)

In [ ]:
X = df[
feature_cols
]

# Target
y = df[
"Future_Risk"
]

In [ ]:
# Split
X_train,X_test,y_train,y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42
)


In [ ]:
# Train model
model = RandomForestClassifier(
n_estimators=300,
max_depth=10,
random_state=42
)

model.fit(
X_train,
y_train
)

In [ ]:
# Test
pred = model.predict(
X_test
)

accuracy = accuracy_score(
y_test,
pred
)

In [ ]:
print(
"Forecast Accuracy:",
accuracy
)